# 1. Carregar o banco de dados.

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1


In [ ]:
import pandas as pd

# Carregando banco de dados
url = r"https://raw.githubusercontent.com/josenalde/machinelearning/refs/heads/main/src/dataset/loan.csv"
loan_df = pd.read_csv(url)

In [ ]:
loan_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


# 2. Removendo colunas desnecessárias.

In [ ]:
#'Credit_History', 'CoapplicantIncome', 'Loan_Amount_Term', 'Property_Area'
df_filtered = loan_df.drop(columns=['Loan_ID'])
df_filtered.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


# 3. Separação da base de dados nos subconjuntos de treino e teste.


In [ ]:
from sklearn.model_selection import train_test_split
X = df_filtered.drop(columns=['Loan_Status'])
y = df_filtered['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Verificando valores faltantes.

In [ ]:
df_filtered.isna().sum()

,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14
Credit_History,50


# 5. Utilizando o SimpleImputer para preencher valores ausentes utilizando a moda.

In [ ]:
from sklearn.impute import SimpleImputer # https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
# Utiliza a Moda
imputer = SimpleImputer(strategy='most_frequent')

# Treina apenas no conjunto de treino e apenas aplica a transformação no conjunto de teste (Evitando Data Leakage)
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 6. Transformação das colunas categóricas para númericas

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

columns = [0,1,3,4]
ordinal_encoder = OrdinalEncoder()

X_train_transformed = X_train_imputed.copy()
X_test_transformed = X_test_imputed.copy()

X_train_transformed[:,columns] = ordinal_encoder.fit_transform(X_train_transformed[:, columns])
X_test_transformed[:,columns] = ordinal_encoder.transform(X_test_transformed[:, columns])

In [ ]:
import numpy as np

dependents_map = {
    '0': 0,
    '1': 1,
    '2': 2,
    '3': 3,
    '3+': 3
}

# Usando np.vectorize para aplicar o mapeamento no NumPy array
map_func = np.vectorize(lambda x: dependents_map.get(str(x), x))

X_train_transformed[:, 2] = map_func(X_train_transformed[:, 2])
X_test_transformed[:, 2] = map_func(X_test_transformed[:, 2])

In [ ]:
X_train_imputed

array([['Male', 'Yes', '0', ..., 360.0, 1.0, 'Semiurban'],
       ['Male', 'Yes', '0', ..., 360.0, 1.0, 'Semiurban'],
       ['Male', 'Yes', '2', ..., 360.0, 1.0, 'Semiurban'],
       ...,
       ['Female', 'No', '0', ..., 360.0, 1.0, 'Urban'],
       ['Female', 'Yes', '0', ..., 240.0, 1.0, 'Semiurban'],
       ['Male', 'Yes', '0', ..., 360.0, 1.0, 'Urban']],
      shape=(491, 11), dtype=object)

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoder = LabelEncoder()
label_encoder.fit_transform(y_train)
label_encoder.transform(y_test)

array([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1])

# 7. Padronização

In [ ]:
from sklearn.preprocessing import StandardScaler
num_cols = [5, 6]
scaler = StandardScaler()

X_train_scaled = X_train_transformed.copy()
X_test_scaled = X_test_transformed.copy()

# Atribui o resultado de volta para as colunas numéricas corretas (num_cols)
X_train_scaled[:, num_cols] = scaler.fit_transform(X_train_scaled[:, num_cols])
X_test_scaled[:, num_cols] = scaler.transform(X_test_scaled[:, num_cols])

In [ ]:
loan_df['Loan_Status'].value_counts()

,count
Loan_Status,
Y,422
N,192


In [ ]:
!pip install imblearn

# 8. Criação dos pipelines aplicando os mesmos pré-processamentos das etapas anteriores


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

dependents_col = ["Dependents"]
categorical_cols = ["Gender", "Married", "Education", "Self_Employed", 'Property_Area']
numerical_cols = ["ApplicantIncome", "LoanAmount",'Credit_History', 'CoapplicantIncome', 'Loan_Amount_Term']

dependents_map = {
    '0': 0,
    '1': 1,
    '2': 2,
    '3': 3,
    '3+': 3
}

def map_dependts(df_or_array):
    s = pd.Series(df_or_array.squeeze()).astype(str)
    return s.map(dependents_map).fillna(0).to_numpy().reshape(-1, 1)

categorical_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])

dependent_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("mapper", FunctionTransformer(map_dependts))
])

numerical_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_cols),
        ("dep", dependent_pipeline, dependents_col),
        ("num", numerical_pipeline, numerical_cols)
    ],
    remainder="drop",
)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
def apply_grid_search(params: dict, model_pipeline):
  cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  grid_search = GridSearchCV(
      estimator=model_pipeline,
      param_grid=params,
      cv=cross_validation,
      scoring="roc_auc",
      verbose=1,
      n_jobs=-1,
      refit=True,
  )

  grid_search.fit(X_train, y_train_encoded)
  y_pred = grid_search.predict(X_test)
  y_proba = grid_search.predict_proba(X_test)[:, 1]

  print(f"Best ROC-AUC (Train): {grid_search.best_score_:.4f}")
  print(f"ROC-AUC (Test): {roc_auc_score(y_test_encoded, y_proba):.4f}")
  print(classification_report(y_test_encoded, y_pred))

  return grid_search

# Random Forest Classifier

In [ ]:
model_pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ("smote", SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_params = {
    "classifier__n_estimators": [50, 100, 200],
    'classifier__max_depth': [2, 3, 4],
    'classifier__min_samples_split': [2, 5, 10],
}

grid_search_rf = apply_grid_search(rf_params, model_pipeline)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best ROC-AUC (Train): 0.7649
ROC-AUC (Test): 0.7426
              precision    recall  f1-score   support

           0       0.70      0.49      0.58        43
           1       0.76      0.89      0.82        80

    accuracy                           0.75       123
   macro avg       0.73      0.69      0.70       123
weighted avg       0.74      0.75      0.73       123



# Resultado com todas as features (exceto o ID):
* ROC-AUC Teste: 0.7426

# Resultado removendo as colunas coapplicantincome, loan_amount_term,credit_history, property_area:
* ROC-AUC Teste: 0.5494

# SVC

In [ ]:
from sklearn.svm import SVC
model_pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ("smote", SMOTE(random_state=42)),
    ('classifier', SVC(random_state=42, probability=True))
])

svc_params = {
    'classifier__C': [0.1, 1.0, 10.0],
    'classifier__kernel': ['linear', 'rbf', 'poly'],
    'classifier__gamma': ['scale', 'auto', 0.1]
}

grid_search_svc = apply_grid_search(svc_params, model_pipeline)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best ROC-AUC (Train): 0.7813
ROC-AUC (Test): 0.7468
              precision    recall  f1-score   support

           0       0.90      0.42      0.57        43
           1       0.76      0.97      0.85        80

    accuracy                           0.78       123
   macro avg       0.83      0.70      0.71       123
weighted avg       0.81      0.78      0.75       123



# Resultado com todas as features (exceto o ID):
* ROC-AUC Teste: 0.7468

# Resultado removendo as colunas coapplicantincome, loan_amount_term,credit_history, property_area:
* ROC-AUC (Test): 0.4938


# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model_pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ("smote", SMOTE(random_state=42)),
    ('classifier', LogisticRegression(random_state=42, solver='saga'))
])

lr_params = {
    'classifier__C': [0.1, 1.0, 10.0],
    'classifier__penalty': ['l1', 'l2', None],
    'classifier__max_iter': [100, 250, 500]
}

grid_search_lr = apply_grid_search(lr_params, model_pipeline)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best ROC-AUC (Train): 0.7605
ROC-AUC (Test): 0.7509
              precision    recall  f1-score   support

           0       0.79      0.51      0.62        43
           1       0.78      0.93      0.85        80

    accuracy                           0.78       123
   macro avg       0.78      0.72      0.73       123
weighted avg       0.78      0.78      0.77       123



# Resultado com todas as features (exceto o ID):
* ROC-AUC Teste: 0.7509

# Resultado removendo as colunas coapplicantincome, loan_amount_term,credit_history, property_area:
* ROC-AUC Teste: 0.5642

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model_pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ("smote", SMOTE(random_state=42)),
    ('classifier', KNeighborsClassifier())
])

knn_params = {
    'classifier__n_neighbors': [3, 5, 11],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search_knn = apply_grid_search(knn_params, model_pipeline)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best ROC-AUC (Train): 0.7535
ROC-AUC (Test): 0.6858
              precision    recall  f1-score   support

           0       0.51      0.42      0.46        43
           1       0.72      0.79      0.75        80

    accuracy                           0.66       123
   macro avg       0.62      0.60      0.61       123
weighted avg       0.65      0.66      0.65       123



# Resultado com todas as features (exceto o ID):
* ROC-AUC Teste: 0.6858

# Resultado removendo as colunas coapplicantincome, loan_amount_term,credit_history, property_area:
* ROC-AUC Teste: 0.4606

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
model_pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ("smote", SMOTE(random_state=42)),
    ('classifier', HistGradientBoostingClassifier())
])

hgb_params = {
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_leaf_nodes': [15, 31, 63],
    'classifier__min_samples_leaf': [10, 20, 50]
}

grid_search_hgb = apply_grid_search(hgb_params, model_pipeline)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best ROC-AUC (Train): 0.7692
ROC-AUC (Test): 0.7253
              precision    recall  f1-score   support

           0       0.67      0.47      0.55        43
           1       0.75      0.88      0.81        80

    accuracy                           0.73       123
   macro avg       0.71      0.67      0.68       123
weighted avg       0.72      0.73      0.72       123



# Resultado com todas as features (exceto o ID):
* ROC-AUC Teste: 0.7692

# Resultado removendo as colunas coapplicantincome, loan_amount_term,credit_history, property_area:
* ROC-AUC Teste: 0.5013

# Em uma etapa adicional foi realizada uma comparação das métricas apresentadas ao treinar os modelos excluindo as features: coapplicantincome, loan_amount_term,credit_history, property_area e ao treinar com todas as features.
# O treinamento com todas as features apresentou um resultado superior para todos os modelos enquanto sem essas features os modelos não são melhores que uma escolha aleátoria.
# Esse resultado indica que as features removidas contém informações importantes para a predição da váriavel alvo.

# Salvamento do modelo escolhido (LogisticRegression)

In [ ]:
# LogisticRegression: maior Roc-AUC no conjunto de testes
import joblib
best_model = grid_search_lr.best_estimator_
joblib.dump(best_model, "lr_all_features.joblib")
joblib.dump(label_encoder, "label_encoder.joblib")

['label_encoder.joblib']